# Rate limits, retries and timeouts

The lookup paces, retries and times out requests on its own. This notebook shows the
settings that control that behaviour and how to batch your own queries politely.

- **Pacing:** before each search call, a source waits `1 / rate` seconds, where `rate`
  comes from `LookupConfig.rate_limits` (requests per second, default 1.0).
- **Retries:** adapters that make HTTP requests through the shared base class retry by
  error category (see the table printed below). Adapters built on third-party clients
  (ChEMBL, UMLS, EBI OLS, Tyto, and the `bioservices`-based EUtils, QuickGO and UniChem)
  rely on those clients instead.
- **Timeouts:** `timeout_per_source` (default 30 s) limits each HTTP request and, when
  several sources are searched in parallel, each source's search.

In [ ]:
import asyncio
import time

from knowledge_lookup import (
    CentralKnowledgeLookup,
    KnowledgeLookupCache,
    KnowledgeSource,
    LookupConfig,
)
from knowledge_lookup.base import DEFAULT_RETRY_STRATEGIES

# delay before retry n = backoff_factor * 2 ** (n - 1), capped at max_delay
for category, (tries, factor, max_delay) in DEFAULT_RETRY_STRATEGIES.items():
    print(f"{category.value:<13} tries={tries} backoff_factor={factor} max_delay={max_delay}s")

## Per-source pacing

Raise the rate for a source you are allowed to query faster. Keys of `rate_limits` are
`KnowledgeSource` members.

In [ ]:
HPO = KnowledgeSource.HPO


async def timed_search(config, query):
    lookup = CentralKnowledgeLookup(config)
    try:
        start = time.perf_counter()
        result = await lookup.search_concepts(query, sources=[HPO], max_results=5)
        return result.total_found, time.perf_counter() - start
    finally:
        await lookup.close()


configs = {
    "default (1 req/s)": LookupConfig(enabled_sources=[HPO]),
    "10 req/s": LookupConfig(enabled_sources=[HPO], rate_limits={HPO: 10.0}),
}
for name, config in configs.items():
    hits, seconds = await timed_search(config, "ataxia")
    print(f"{name:<18} {hits} hits in {seconds:.2f}s")

## Many queries

Every call paces itself, but concurrent calls to the same source can still overlap. Bound
concurrency with a semaphore when you run a batch.

In [ ]:
terms = ["asthma", "ataxia", "seizure", "microcephaly"]
lookup = CentralKnowledgeLookup(LookupConfig(enabled_sources=[HPO], rate_limits={HPO: 2.0}))
semaphore = asyncio.Semaphore(2)  # at most two searches at a time


async def search(term):
    async with semaphore:
        result = await lookup.search_concepts(term, sources=[HPO], max_results=3)
        return term, [concept.primary_label for concept in result.concepts]


for term, labels in await asyncio.gather(*(search(term) for term in terms)):
    print(f"{term}: {labels}")
await lookup.close()

## Timeouts

When several sources are searched together, a source that exceeds `timeout_per_source`
is reported in `result.errors` and the other sources still return their concepts.
Searching a single source runs without that per-source guard, so the adapter's HTTP
timeout applies instead.

The pacing delay counts towards the timeout. Below, HPO waits 0.1 s before its request,
while Ensembl waits 1 s and then needs to answer its (usually slow) gene search within
the remaining half second, so it normally times out.

In [ ]:
config = LookupConfig(
    enabled_sources=[KnowledgeSource.HPO, KnowledgeSource.ENSEMBL],
    rate_limits={KnowledgeSource.HPO: 10.0},
    timeout_per_source=1.5,
)
lookup = CentralKnowledgeLookup(config)
result = await lookup.search_concepts("TP53", max_results=10)
print("Succeeded:", list(result.sources_succeeded))
print("Errors:", result.errors)
await lookup.close()

## Caching repeated queries

`CentralKnowledgeLookup` does not cache search results (only the UniChem adapter caches
its internal lookups). To avoid repeating the same requests, cache results in your own
code, for example with `KnowledgeLookupCache`.

In [ ]:
cache = KnowledgeLookupCache(default_ttl=3600)  # in memory; pass disk_cache_dir= to persist
lookup = CentralKnowledgeLookup(LookupConfig(enabled_sources=[HPO]))


async def cached_search(term):
    labels = cache.get(term, namespace="hpo")
    if labels is None:
        result = await lookup.search_concepts(term, sources=[HPO], max_results=5)
        labels = [concept.primary_label for concept in result.concepts]
        cache.set(term, labels, namespace="hpo")
    return labels


for attempt in (1, 2):
    start = time.perf_counter()
    labels = await cached_search("ataxia")
    print(f"call {attempt}: {len(labels)} labels in {time.perf_counter() - start:.2f}s")
await lookup.close()

## Summary

- Use API keys where a source offers higher limits with one.
- Tune `rate_limits` per source instead of adding sleeps around every call.
- Bound concurrency for batches, and cache repeated queries yourself.
- Keep `timeout_per_source` realistic for slow sources such as Ensembl.

Next: [04 Error handling](04-error-handling.ipynb).